# 3. OpenAI Debugging Tutor

### Build an OpenAI debugging tutor and compare how context management changes token use, cost, latency, and the next reply.

This notebook uses `gpt-4o-mini`, a tutor prompt, and a chat interface with an experiment log.
An API model can handle longer workflows than a small local model, but the context window is still finite.
As chat history grows, more input tokens are sent with each request.
That changes cost, latency, and how much space remains for the next user message and the tutor instructions.

This notebook studies one response to that problem: summary compression.
Older turns can be compressed into a shorter note while recent turns stay in full form.

**Focus**
- how much raw chat history is kept in the prompt
- how summary compression changes input tokens, cost, and latency
- how much detail survives after older turns are compressed
- whether a shorter summary still preserves enough context for a useful next hint



In [8]:
# %pip -q install gradio openai python-dotenv

import os  # reads environment variables and file paths
import time  # measures response latency
import gradio as gr  # builds the web interface
from openai import OpenAI  # calls the OpenAI API
from dotenv import load_dotenv  # loads variables from a .env file


## 2. Connect to OpenAI

This notebook uses one API model: `gpt-4o-mini`.
Keeping the model fixed makes the experiment easier to read because the main variable is context management, not model choice.
The notebook can then compare prompt length and history strategy without mixing in model-to-model differences.

This cell loads `OPENAI_API_KEY`, creates the OpenAI client, and prints a short status message.
If the key is missing, the notebook can still open, but model calls will not work until the key is available.


In [9]:
api_client = None
model_name = "gpt-4o-mini"

load_dotenv(".env")
api_key = os.getenv("OPENAI_API_KEY")
print("OPENAI_API_KEY:", "loaded" if api_key else "NOT FOUND")

if api_key:
    api_client = OpenAI(api_key=api_key)
    print("OpenAI client ready.")
else:
    print("Add OPENAI_API_KEY to your environment or .env file and rerun this cell.")

print("Using model:", model_name)


OPENAI_API_KEY: loaded
OpenAI client ready.
Using model: gpt-4o-mini


## 3. Blueprint

Before the app runs, the tutor needs a prompt and a set of repeatable questions.

- `system_prompt` defines the tutoring style
- `test_cases` stores one short starter prompt and two follow-up details for each case

The dropdown loads only the starter prompt.
That first message is intentionally short.
The `Next Turn` button adds `detail_1` and then `detail_2`.
That makes the workflow closer to a real debugging conversation, where the question becomes more specific after the first reply.

This notebook focuses on what happens when conversation history gets longer.
A longer prompt means more input tokens.
Input tokens matter because API pricing is based on token counts, so a longer request costs more than a shorter request.
A longer request also means the model has more text to process before it can answer.
Research on long-context models further shows that information inside a long prompt is not always used equally well, so simply adding more text is not the same as making the next answer better.

Instead of sending every earlier turn in full, this notebook compresses older turns into a shorter summary.
That is the main context-management idea to study here.
The notebook is designed to show what is kept exactly, what is compressed, and how that choice affects the next response.




In [10]:
app_title = "AI Debugging Tutor — OpenAI Responses API"
app_desc = (
    "An OpenAI debugging tutor with gpt-4o-mini, "
    "summary compression, and a simple experiment log."
)

system_prompt = '''
You are a debugging tutor.

Rules:
- Do not give the final answer or full corrected code.
- Ask one short question or suggest one short check.
- Keep each reply short and focused.
- Stay on one issue at a time.
'''

# Each case starts with a short bug report.
# Add detail_1 and detail_2 in later turns.
test_cases = {
    "1. printed value failed the test [Easy]": {
        "starter": """scores = [78, 85, 92]
average_score = sum(scores) / len(scores)
print(int(average_score))

This failed one grader test.""",
        "detail_1": "The expected output is 85.0.",
        "detail_2": "The grader says the value is right, but the printed format is wrong."
    },
    "2. table columns failed the test [Easy]": {
        "starter": """cookie_summary = Table().with_columns(
    'Type', ['Chocolate chip', 'Red velvet'],
    'Amount remaining', [15, 8]
).select('Type', 'Amount remaining')
print(cookie_summary)

I thought this table was fine, but it still failed.""",
        "detail_1": "The expected column order is Amount remaining, Type.",
        "detail_2": "The grader says the values are correct and only the column order is wrong."
    },
    "3. sample changed between runs [Mid]": {
        "starter": """student_scores = pd.DataFrame({
    'student_id': [101, 102, 103, 104, 105, 106],
    'score': [88, 91, 75, 84, 93, 79]
})

first_sample = student_scores.sample(n=3)
second_sample = student_scores.sample(n=3)

print(first_sample)
print()
print(second_sample)

The sample changes every time I run this.""",
        "detail_1": "I need the same rows each time.",
        "detail_2": "The hidden test says the result is not reproducible."
    },
    "4. filtered rows still failed [Mid]": {
        "starter": """student_scores = pd.DataFrame({
    'student_id': [101, 102, 103, 104, 105],
    'score': [88, 91, 75, 84, 93]
})

top_scores = student_scores[student_scores['score'] >= 90]
print(top_scores)

These look like the right rows, but the test still fails.""",
        "detail_1": "The printed index is 1 and 4.",
        "detail_2": "The expected index is 0 and 1."
    },
    "5. grouped summary lost one category [Hard]": {
        "starter": """attendance = pd.DataFrame({
    'section': ['A', 'A', None, 'B'],
    'present': [1, 0, 1, 1]
})

attendance_rate = attendance.groupby('section')['present'].mean()
print(attendance_rate)

One group disappeared from the result.""",
        "detail_1": "The missing rows have blank values in the section column.",
        "detail_2": "The hidden test expects the blank section values to appear as their own group."
    },
}

pricing_input_per_million = 0.15
pricing_output_per_million = 0.60


## 4. State

This app keeps two pieces of state while it runs.

- `run_log` stores recent experiment results
- `history_summary` stores compressed notes from older turns

These two values serve different purposes.
The log helps compare runs.
The summary keeps older context in a shorter form so fewer raw tokens need to be sent on later turns.
Together, these two state variables let the notebook show both the model output and the context-management decision behind that output.


In [11]:
run_log = []
history_summary = ""

## 5. Helper functions

This section contains the logic that builds the message list, summarizes older turns, sends the request, and records the result.
The code in this section is where the notebook turns the idea of summary compression into a real prompt-building process.

The context strategy has two layers.

1. recent turns stay in full form
2. older turns can be reduced to a short summary

The two bottom panels make that process visible.
`Context` shows the exact request sent to the API.
`History Summary` shows the compressed notes that stand in for older conversation history.
Together, those panels make it easier to connect context length to token use, cost, latency, and the next reply.


In [12]:
def clean_message(message):
    role = message.get("role", "user")
    content = message.get("content", "")

    if isinstance(content, str):
        text = content
    elif isinstance(content, dict):
        text = content.get("text", str(content))
    elif isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict) and "text" in item:
                parts.append(str(item["text"]))
            else:
                parts.append(str(item))
        text = "".join(parts)
    else:
        text = str(content)

    if role == "assistant":
        text = text.split("\n\n`Time: ")[0]

    return {"role": role, "content": text}


def build_summary_source(old_history):
    lines = []
    for message in old_history:
        clean_item = clean_message(message)
        lines.append(f"[{clean_item['role'].upper()}] {clean_item['content']}")
    return "\n".join(lines)


def split_history(history, memory_turns):
    keep_messages = int(memory_turns) * 2

    if keep_messages <= 0:
        return list(history), []

    if len(history) <= keep_messages:
        return [], list(history)

    old_history = list(history[:-keep_messages])
    recent_history = list(history[-keep_messages:])
    return old_history, recent_history


def count_text_tokens(text):
    return max(1, len(str(text).split()))


def extract_output_text(response):
    text = getattr(response, "output_text", None)
    if text:
        return text

    output_items = getattr(response, "output", None) or []
    parts = []
    for item in output_items:
        content_items = getattr(item, "content", None) or []
        for content_item in content_items:
            maybe_text = getattr(content_item, "text", None)
            if maybe_text:
                parts.append(maybe_text)
    return "".join(parts)


def summarize_history(old_history):
    global history_summary

    if not old_history:
        return

    if api_client is None:
        history_summary = "OpenAI client is not ready, so compressed history is unavailable."
        return

    try:
        response = api_client.responses.create(
            model=model_name,
            instructions=(
                "Summarize this older conversation in 4 short bullet points. "
                "Keep the user's important mistakes and useful facts. "
                "Do not answer the user."
            ),
            input=[{"role": "user", "content": build_summary_source(old_history)}],
            temperature=0.2,
            max_output_tokens=160,
        )
        history_summary = (extract_output_text(response) or "").strip()
    except Exception as error:
        history_summary = "History compression failed: " + str(error)


def build_messages(system_prompt, history, memory_turns):
    global history_summary

    messages = [{"role": "system", "content": system_prompt.strip()}]

    if int(memory_turns) <= 0:
        history_summary = ""
        return messages

    old_history, recent_history = split_history(history, memory_turns)

    if old_history:
        summarize_history(old_history)
    else:
        history_summary = ""

    if history_summary:
        messages.append({
            "role": "system",
            "content": "Conversation summary from older turns:\n" + history_summary,
        })

    for message in recent_history:
        messages.append(clean_message(message))

    return messages


def build_context_text(messages):
    parts = []
    for message in messages:
        parts.append(f"[{message['role'].upper()}]")
        parts.append(str(message["content"]))
        parts.append("-" * 40)
    return "\n".join(parts)


def build_history_summary(memory_turns):
    if int(memory_turns) <= 0:
        return "Memory is off. Only the current user message is sent."
    if history_summary == "":
        return "No compressed history yet."
    return history_summary


def calculate_cost_usd(input_tokens, output_tokens):
    input_cost = (int(input_tokens) / 1_000_000) * pricing_input_per_million
    output_cost = (int(output_tokens) / 1_000_000) * pricing_output_per_million
    return input_cost + output_cost


def generate_response(messages, temperature, max_tokens):
    if api_client is None:
        final_text = "OpenAI client is not ready. Load OPENAI_API_KEY and rerun the connection cell."
        return final_text, None, count_text_tokens(final_text)

    try:
        response = api_client.responses.create(
            model=model_name,
            input=messages,
            temperature=float(temperature),
            max_output_tokens=min(int(max_tokens), 2048),
        )
        final_text = extract_output_text(response) or ""
        if final_text == "":
            final_text = "[No text returned]"

        usage = getattr(response, "usage", None)
        prompt_tokens = getattr(usage, "input_tokens", None) if usage is not None else None
        output_tokens = getattr(usage, "output_tokens", None) if usage is not None else None
        if output_tokens is None:
            output_tokens = count_text_tokens(final_text)

        return final_text, prompt_tokens, output_tokens
    except Exception as error:
        final_text = "Generation failed: " + str(error)
        return final_text, None, count_text_tokens(final_text)


def run_chatbot(user_input, system_prompt, memory_turns, temperature, max_tokens, test_case, history):
    global run_log

    if history is None:
        history = []

    messages = build_messages(system_prompt, history, memory_turns)

    if len(history) == 0 and test_case != "Free Typing" and test_case in test_cases:
        if user_input.strip():
            user_message = user_input.strip()
        else:
            user_message = test_cases[test_case]["starter"]
    else:
        user_message = user_input.strip()

    history_text = build_history_summary(memory_turns)

    if user_message == "":
        log_rows = []
        for record in run_log[-10:]:
            log_rows.append([
                record["run"],
                record["test_case"],
                record["memory_turns"],
                record["input_tokens"],
                record["output_tokens"],
                record["cost"],
                record["latency"],
            ])
        return history, "", history_text, log_rows, "", "**Status:** Waiting for input"

    messages.append({"role": "user", "content": user_message})
    context_text = build_context_text(messages)

    display_user = user_input.strip()
    if display_user == "":
        if len(history) == 0 and test_case != "Free Typing" and test_case in test_cases:
            display_user = test_cases[test_case]["starter"]
        else:
            display_user = "[" + test_case + "]"

    history = list(history)
    history.append({"role": "user", "content": display_user})
    history.append({"role": "assistant", "content": ""})

    start_time = time.perf_counter()
    final_text, prompt_tokens, output_tokens = generate_response(messages, temperature, max_tokens)
    latency = round(time.perf_counter() - start_time, 2)

    input_tokens = prompt_tokens
    if input_tokens is None:
        input_tokens = count_text_tokens(context_text)

    cost_usd = calculate_cost_usd(input_tokens, output_tokens)
    cost_text = f"${cost_usd:.6f}"

    badge = (
        f"\n\n`Model: {model_name}"
        f" | Time: {latency}s"
        f" | Input tokens: {input_tokens}"
        f" | Output tokens: {output_tokens}"
        f" | Cost: {cost_text}`"
    )
    history[-1]["content"] = final_text + badge

    run_log.append({
        "run": len(run_log) + 1,
        "test_case": test_case,
        "memory_turns": int(memory_turns),
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "cost": cost_text,
        "latency": latency,
    })

    log_rows = []
    for record in run_log[-10:]:
        log_rows.append([
            record["run"],
            record["test_case"],
            record["memory_turns"],
            record["input_tokens"],
            record["output_tokens"],
            record["cost"],
            record["latency"],
        ])

    return history, context_text, build_history_summary(memory_turns), log_rows, "", "**Status:** Done"


def clear_chat():
    global history_summary
    global run_log
    history_summary = ""
    run_log = []
    return [], "", "No compressed history yet.", [], "", "**Status:** Ready"


def load_test_case(test_case):
    if test_case == "Free Typing":
        return "", "**Status:** Ready", 0
    return test_cases[test_case]["starter"], "**Status:** Loaded starter prompt for " + test_case, 0


def load_next_turn(test_case, detail_stage, current_text):
    if test_case == "Free Typing" or test_case not in test_cases:
        return current_text, "**Status:** Next Turn is only available for a test case.", detail_stage

    if detail_stage <= 0:
        next_text = test_cases[test_case]["detail_1"]
        return next_text, "**Status:** Loaded detail 1 for " + test_case, 1

    if detail_stage == 1:
        next_text = test_cases[test_case]["detail_2"]
        return next_text, "**Status:** Loaded detail 2 for " + test_case, 2

    return current_text, "**Status:** No more saved details for " + test_case, detail_stage


def run_chatbot_ui(user_input, system_prompt, memory_turns, temperature, max_tokens, test_case, chat_state):
    history, context_text, history_text, log_rows, next_input, status_message = run_chatbot(
        user_input,
        system_prompt,
        memory_turns,
        temperature,
        max_tokens,
        test_case,
        chat_state,
    )
    return history, context_text, history_text, log_rows, next_input, status_message, history


def clear_chat_ui():
    history, context_text, history_text, log_rows, next_input, status_message = clear_chat()
    return history, context_text, history_text, log_rows, next_input, status_message, history, 0





## 6. Build the interface

This section places the tutor inside a Gradio app.
The interface is arranged to keep the experiment readable while the notebook is running.

The left side is the active chat.
The right side contains the experiment controls.
The bottom panels show the hidden mechanics: the exact request, the compressed history, and the recent log.

That layout keeps the main context-management tradeoff on screen.
The reply shows what the model did.
The lower panels show how much prompt text was sent to get that reply.
The log turns each interaction into a comparable run with measurements instead of a one-time example.


In [24]:
ui_test_cases = ["Free Typing"] + list(test_cases.keys())

with gr.Blocks(title=app_title) as web_app:
    gr.Markdown("# " + app_title)
    gr.Markdown("*" + app_desc + "*")
    gr.Markdown("**Model:** " + model_name)

    chat_state = gr.State([])
    detail_stage_state = gr.State(0)

    with gr.Row():
        with gr.Column(scale=7):
            chat_window = gr.Chatbot(label="Chat", height=433)
            with gr.Row():
                user_input_box = gr.Textbox(
                    show_label=False,
                    placeholder="Type your debugging question here...",
                    lines=8,
                    scale=8,
                )
                with gr.Column(scale=1, min_width=60):
                    ask_button = gr.Button("Ask", variant="primary")
                    next_turn_button = gr.Button("Next Turn", variant="primary")
                    clear_button = gr.Button("New Chat")

        with gr.Column(scale=4):
            test_case_dropdown = gr.Dropdown(
                choices=ui_test_cases,
                value="Free Typing",
                label="Test Case",
            )
            with gr.Accordion("Model Settings", open=True):
                memory_turns_slider = gr.Slider(0, 8, step=1, value=1, label="Memory Turns")
                temperature_slider = gr.Slider(0.0, 1.0, step=0.1, value=0.2, label="Temperature")
                max_tokens_slider = gr.Slider(64, 2048, step=64, value=512, label="Max Tokens")

            with gr.Accordion("Edit Prompt", open=False):
                system_prompt_box = gr.Textbox(label="System Prompt", value=system_prompt, lines=8)

    with gr.Row():
        context_box = gr.Textbox(label="Context", value="", lines=16, interactive=False)
        history_box = gr.Textbox(
            label="History Summary",
            value="No compressed history yet.",
            lines=16,
            interactive=False,
        )

    log_table = gr.Dataframe(
        headers=["Run", "Test Case", "Memory Turns", "Input", "Output", "Cost (USD)", "Latency"],
        datatype=["number", "str", "number", "number", "number", "str", "number"],
        row_count=10,
        column_count=(7, "fixed"),
        column_widths=["8%", "24%", "12%", "12%", "12%", "16%", "16%"],
        interactive=False,
        label="Experiment Log",
    )

    status_box = gr.Markdown("**Status:** Ready")

    inputs = [
        user_input_box,
        system_prompt_box,
        memory_turns_slider,
        temperature_slider,
        max_tokens_slider,
        test_case_dropdown,
        chat_state,
    ]

    outputs = [
        chat_window,
        context_box,
        history_box,
        log_table,
        user_input_box,
        status_box,
        chat_state,
    ]

    test_case_dropdown.change(load_test_case, test_case_dropdown, [user_input_box, status_box, detail_stage_state])
    next_turn_button.click(load_next_turn, [test_case_dropdown, detail_stage_state, user_input_box], [user_input_box, status_box, detail_stage_state])
    ask_button.click(run_chatbot_ui, inputs, outputs)
    user_input_box.submit(run_chatbot_ui, inputs, outputs)
    clear_button.click(clear_chat_ui, None, outputs + [detail_stage_state])






## 7. Run the app

This cell launches the OpenAI web app.

Work with one case for several turns instead of changing cases too quickly.
The dropdown loads only the short starter prompt.
The same test case has two extra details that the `Next Turn` button can add later.

A useful workflow is:

1. Pick one test case and run the starter prompt.
2. Read the tutor reply and note what is still vague.
3. Click `Next Turn` once to add `detail_1`.
4. Compare the second reply with the first one.
5. Click `Next Turn` again to add `detail_2`.
6. Clear the chat.
7. Run the same case again with `Memory Turns` set to `0`, then `1` or `2`.
8. Compare the `History Summary` panel and the log.

**Look for**
- whether each added detail makes the next hint more concrete
- whether older details survive in the summary
- how input tokens, cost, and latency change when more or less context is sent to the model
- whether the compressed summary keeps enough detail for the next hint to stay useful




In [25]:
web_app.launch(share=True)

* Running on local URL:  http://127.0.0.1:7870
* Running on public URL: https://77cb347d1a2ecb616c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Summary

This notebook treats long chat history as a context-management problem.
A longer conversation means more input tokens, and more input tokens increase request cost.
A longer prompt also gives the model more text to process before producing the next answer.
Instead of carrying every earlier turn forward in full, the notebook compresses older turns into shorter notes.

The main topic here is prompt budgeting.
A request has limited context space, and that space has a cost.
Summary compression is one way to preserve important earlier information while using fewer raw chat tokens.
